All raw data used for the project are available for viewing at https://drive.google.com/drive/folders/1zYKFfSXptnHTim8ERtAOu1GdgunQIPzA?usp=sharing

Installations

In [9]:
!pip install -U pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 61.5 MB/s eta 0:00:00


Load datasets into the notebook

In [10]:
# Mounting procedure adapted from https://colab.research.google.com/notebooks/io.ipynb#scrollTo=RWSJpsyKqHjH
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [11]:
import dask.bag as db
import pypdfium2 as pdfium

In [12]:
# @title book file names
book_file_names = ['babicka_bozena_nemcova',
'windows8_redakce_businessIT_a_partneri',
'cesky_rozhlas-historie_eva_jesutova_a_kolektiv',
'flvek_05_alois_jirasek',
'flvek_4_alois_jirasek',
'flvek_03_alois_jirasek',
'flvek_02_alois_jirasek',
'flvek_01_alois_jirasek',
'jihoslovanske_jazyky_pavel_krejci',
'nase_nynejsi_krise_tomas_garrigue_masaryk',
'lucerna_alois_jirasek',
'matka_karel_capek',
'basne_josef_vaclav_sladek',
'dalimilova_kronika_dalimil',
'obycejny_zivot_karel_capek',
'povetron_karel_capek',
'hordubal_karel_capek',
'noc_na_karlstejne_jaroslav_vrchlicky',
'pisne_kosmicke_jan_neruda',
'domaci_kucharka_magdalena_dobromila_rettigova',
'hovory_s_tg_masarykem_karel_capek',
'dramaticke_zlomky_karel_hynek_macha',
'povidani_o_pejskovi_a_kocicce_josef_capek',
'obrazy_z_dejin_naroda_ceskeho_iii_vladislav_vancura',
'obrazy_z_dejin_naroda_ceskeho_ii_vladislav_vancura',
'obrazy_z_dejin_naroda_ceskeho_i_vladislav_vancura',
'rur_karel_capek',
'bylo_nas_pet_karel_polacek',
'mistr_kampanus_zikmund_winter',
'konec_starych_casu_vladislav_vancura',
'ballady_a_romance_jan_neruda',
'vec_makropulos_karel_capek',
'krakatit_karel_capek',
'filosofska_historie_alois_jirasek',
'stare_povesti_ceske_alois_jirasek',
'devatero_pohadek_karel_capek',
'kosmuw_letopis_cesky_kosmas',
'tezka_hodina_jiri_wolker',
'host_do_domu_jiri_wolker',
'nova_evropa_tomas_garrigue_masaryk',
'sedm_let_v_jizni_africe_iv_emil_holub',
'sedm_let_v_jizni_africe_iii_emil_holub',
'sedm_let_v_jizni_africe_druha_cesta_emil_holub',
'sedm_let_v_jizni_africe_prvni_cesta_emil_holub',
'bila_nemoc_karel_capek',
'rozmarne_leto_vladislav_vancura',
'maj_karel_hynek_macha',
'kytice_karel_jaromir_erben',
'osudy_dobreho_vojaka_svejka_jaroslav_hasek_volumes3and4',
'osudy_dobreho_vojaka_svejka_jaroslav_hasek_volumes1and2',
'broucci_jan_karafiat',
]

In [13]:
def book_loader(filename):
  document = pdfium.PdfDocument('/content/drive/My Drive/Colab Notebooks/BigDataProject/books/' + filename + '.pdf')

  version_marker_found = 0
  book_started = 0
  text = ''
  for page in document:
    # extract text from the page
    textpage = page.get_textpage()
    extractedtext = textpage.get_text_bounded()

    # the following two conditions ensure that the material attached to the book that is not a part of the original text is skipped (for example the cover page, info about publication etc.)
    if version_marker_found == 0 and 'verze' in extractedtext.lower():
      version_marker_found = 1
      print('version marker found')

    # after the version marker is found (indicating the last page of added material), the next page is checked for containing the contents (obsah) of the book which can also be skipped.
    elif version_marker_found and book_started == 0:
      if 'obsah' not in extractedtext.lower():
        book_started = 1

    if book_started:
      text += extractedtext

  return filename, text

# a function that removes every expression in a list from the given string
def clean_text(text: str, expr: list[str]) -> str:
  for e in expr:
    text = text.replace(e, '')
    print(e)

  return text



In [23]:
wiki_expr_to_clean = ['\n', '\r', ',', '-', '–', '—', ';', '“', '0', '1', '2', '3', '4', '5', '6','7','8','9','\x02', '(', ')', '„', '...', '•']

In [31]:
mystr = 'hello there'
print('lle' in mystr)

False


In [36]:
# load the books
books = db.from_sequence(book_file_names).map(book_loader)

# load the czech wikipedia
wiki = db.read_text('/content/drive/My Drive/Colab Notebooks/BigDataProject/wiki/extracted/extracted/*/*').filter(lambda x: x[:4] != '<doc' and '__NOEDITSECTION__' not in x and x != '</doc>').map(lambda x: ('wiki', text_cleaner(x, wiki_expr_to_clean)))

In [37]:
print(wiki.take(100))

(('wiki', 'Hlavní strana'), ('wiki', ''), ('wiki', 'internetové encyklopedii kterou může .&ltbr&gtČeská Wikipedie má nyní .'), ('wiki', '&ltbr&gt&ltbr&gt'), ('wiki', '   '), ('wiki', '  '), ('wiki', 'Ostatní projekty'), ('wiki', 'Další informace…'), ('wiki', '  '), ('wiki', '. v minulosti'), ('wiki', ''), ('wiki', '</doc>'), ('wiki', 'Astronomie'), ('wiki', ''), ('wiki', 'Astronomie řecky αστρονομία z άστρον astron hvězda a νόμος nomos zákon česky též hvězdářství je věda která se zabývá jevy za hranicemi zemské atmosféry. Zvláště tedy výzkumem vesmírných těles jejich soustav různých dějů ve vesmíru i vesmírem jako celkem.'), ('wiki', 'Historie astronomie.'), ('wiki', 'Antika.'), ('wiki', 'Astronomie se podobně jako další vědy začala rozvíjet ve starověku. Na území Babylonie však nebylo k popisu používáno již vynalezené geometrie grafy. První se z astronomie rozvíjela astrometrie zabývající se měřením poloh hvězd a planet na obloze. Tato oblast astronomie měla velký význam pro navigaci.

In [18]:
# clean the text
book_expr_to_clean = ['\n', '\r', ',', '-', '–', '—', ';', '“', '0', '1', '2', '3', '4', '5', '6','7','8','9','\x02', '(', ')', '„', '...']
books = books.map(lambda x: (x[0], text_cleaner(x[1], book_expr_to_clean)))
wiki = wiki.map(lambda x: ('wiki', ))

In [ ]:
# todo: expand tzv. to takzvane

In [ ]:
print(b.take(1))